# Wine Quality — Analysis Notebook

Generated by `scripts/build_notebook.py` and executed end to end, so every table and
chart below is a reproducible pipeline artefact.

1. Load and validate both source files
2. De-duplicate and document the cleaning step
3. Explore quality, chemistry and wine type
4. Review the regression and classification results


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd

from wine import config, data, eda
from wine.clean import clean
from wine.features import add_targets

plt.rcParams["figure.figsize"] = (10, 4.5)
pd.set_option("display.max_columns", 50)


## 1. Load and validate

In [2]:
raw = data.load_raw()
validation = data.validate(raw)
print(f"rows={validation['rows']:,}  columns={validation['columns']}")
print("rows by type:", validation["rows_by_type"])
print("exact duplicates:", validation["duplicate_rows"])
raw.head()


rows=6,497  columns=13
rows by type: {'white': 4898, 'red': 1599}
exact duplicates: 1177


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,wine_type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red


## 2. Cleaning

In [3]:
cleaned, log = clean(raw)
summary = pd.DataFrame(
    {
        "rows_in": [log["rows_in"]],
        "duplicates_removed": [log["duplicates_removed"]],
        "rows_out": [log["rows_out"]],
        "good_wine_rate": [log["good_wine_rate"]],
    }
)
summary


,rows_in,duplicates_removed,rows_out,good_wine_rate
0,6497,1177,5320,0.1897


Duplicate rows are repeat lab measurements. Keeping them would leak the same wine into
both training and validation folds, so they are removed once and reported.


## 3. Quality distribution

In [4]:
counts = cleaned["quality"].value_counts().sort_index()
share = (cleaned.groupby("wine_type", observed=True)["quality"]
         .value_counts(normalize=True).unstack().fillna(0).round(3))
share


quality,3,4,5,6,7,8,9
wine_type,,,,,,,
red,0.007,0.039,0.425,0.394,0.123,0.013,0.0
white,0.005,0.039,0.297,0.451,0.174,0.033,0.001


## 4. Chemistry by quality

In [5]:
enriched = add_targets(cleaned)
enriched.groupby("quality")[
    ["alcohol", "volatile acidity", "sulphates", "density"]
].median().round(3)


,alcohol,volatile acidity,sulphates,density
quality,,,,
3,10.15,0.415,0.505,0.996
4,10.10,0.385,0.490,0.995
5,9.60,0.340,0.510,0.996
6,10.50,0.270,0.520,0.994
7,11.50,0.280,0.520,0.992
8,12.20,0.280,0.490,0.992
9,12.50,0.270,0.460,0.990


## 5. Correlations with quality

In [6]:
import seaborn as sns

numeric = enriched.select_dtypes("number")
corr = numeric.corr()[config.TARGET].drop(config.TARGET).sort_values()
corr.to_frame("correlation_with_quality").round(3)


,correlation_with_quality
density,-0.326
volatile acidity,-0.265
chlorides,-0.202
sulphates_alcohol,-0.127
is_red,-0.115
sugar_alcohol_ratio,-0.100
acidity_ph,-0.097
total_acidity,-0.096
bound_so2,-0.084
fixed acidity,-0.080


## 6. Model results

Read from `reports/metrics.json`, written by `python run.py train`.

In [7]:
import json

metrics_path = ROOT / "reports" / "metrics.json"
metrics = json.loads(metrics_path.read_text()) if metrics_path.exists() else {}

if metrics:
    reg = metrics["regression"]
    clf = metrics["classification"]
    print(f"Regression: {reg['selected_model']}  RMSE={reg['rmse']:.3f}  R2={reg['r2']:.3f}")
    print(f"Classification: {clf['selected_model']}  AUC={clf['roc_auc']:.3f}  F1={clf['f1']:.3f}")
    display(pd.DataFrame(reg["cv_comparison"]).set_index("model").round(4))
    display(pd.DataFrame(clf["cv_comparison"]).set_index("model").round(4))
else:
    print("Run `python run.py train` first to populate metrics.")


Regression: random_forest  RMSE=0.660  R2=0.421
Classification: random_forest  AUC=0.874  F1=0.609


,rmse_mean,rmse_std,mae_mean,r2_mean,fit_seconds
model,,,,,
random_forest,0.6985,0.0187,0.5380,0.3730,7.30
gradient_boosting,0.7009,0.0219,0.5459,0.3685,3.96
ridge,0.7333,0.0217,0.5657,0.3089,0.22


,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std,f1_mean,f1_std,recall_mean,recall_std,precision_mean,precision_std,fit_seconds
model,,,,,,,,,,,
random_forest,0.8503,0.0136,0.5737,0.0336,0.5090,0.0224,0.4263,0.0206,0.6330,0.0388,2.94
gradient_boosting,0.8395,0.0087,0.5472,0.0291,0.4421,0.0094,0.3532,0.0150,0.5934,0.0285,5.02
logistic_regression,0.8316,0.0122,0.5213,0.0183,0.5362,0.0205,0.7781,0.0347,0.4093,0.0182,0.06


## 7. Takeaways

In [8]:
if metrics:
    for task in ("regression", "classification"):
        top = pd.DataFrame(metrics[task]["feature_importance"]).head(5)
        print(f"\n{task.title()} drivers")
        for row in top.itertuples():
            print(f"  - {row.feature:<24} {row.importance:.4f}")



Regression drivers
  - alcohol                  0.2113
  - volatile acidity         0.0958
  - density                  0.0807
  - free_so2_ratio           0.0667
  - free sulfur dioxide      0.0581

Classification drivers
  - alcohol                  0.1823
  - density                  0.0955
  - chlorides                0.0676
  - volatile acidity         0.0666
  - citric acid              0.0553
